In [1]:
!pip install transformers torch pandas tqdm -q

In [11]:
import torch
import os
import ast
import pandas as pd
import numpy as np
from itertools import permutations
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
train_df = pd.read_csv('/content/drive/MyDrive/sentence_prediction/train.csv')
test_df  = pd.read_csv('/content/drive/MyDrive/sentence_prediction/test.csv')

print(f"train: {len(train_df)}개")
print(f"test : {len(test_df)}개")
print()
print("샘플 확인:")
print(train_df.head(2))

train: 7351개
test : 1780개

샘플 확인:
           ID                                         sentence_0  \
0  TRAIN_0000                 블록체인 기술은 투표 과정의 투명성을 크게 향상시킬 수 있다.   
1  TRAIN_0001  줄거리 자동 생성의 인공지능 알고리즘은 대량의 텍스트 데이터를 분석하여 핵심 정보를...   

                                          sentence_1  \
0  이러한 특성은 유권자들에게 신뢰를 제공하며, 민주적 참여를 촉진하는 데 기여할 수 있다.   
1     결과적으로, 이러한 기술은 사용자에게 신속하고 효율적인 정보 전달을 가능하게 한다.   

                                          sentence_2  \
0  결과적으로 블록체인 기반의 투표 시스템은 공정하고 신뢰할 수 있는 선거 환경을 조성...   
1     생성된 줄거리는 원본 텍스트의 의미를 유지하면서도 간결하게 요약된 형태로 제공된다.   

                                          sentence_3  answer_0  answer_1  \
0       각 투표는 변경 불가능한 기록으로 저장되어 조작의 가능성을 원천적으로 차단한다.         0         3   
1  이 알고리즘은 자연어 처리 기술을 활용하여 문맥을 이해하고, 주요 사건과 등장인물을...         0         3   

   answer_2  answer_3  
0         1         2  
1         2         1  


In [5]:
MODEL_NAME = "kakaocorp/kanana-nano-2.1b-base"

print(f"로드 중: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print("로드 완료!")

로드 중: kakaocorp/kanana-nano-2.1b-base


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/692 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/4.17G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

로드 완료!


In [6]:
SEPARATOR = " "     # 실험 1-1: 공백
# SEPARATOR = "\n"  # 실험 1-2: 줄바꿈
# SEPARATOR = ". "  # 실험 1-3: 마침표

print(f"현재 연결방식: {repr(SEPARATOR)}")

sample = train_df.iloc[0]
sentences = [sample['sentence_0'], sample['sentence_1'],
             sample['sentence_2'], sample['sentence_3']]
print("\n연결 결과 예시:")
print(SEPARATOR.join(sentences))

현재 연결방식: ' '

연결 결과 예시:
블록체인 기술은 투표 과정의 투명성을 크게 향상시킬 수 있다. 이러한 특성은 유권자들에게 신뢰를 제공하며, 민주적 참여를 촉진하는 데 기여할 수 있다. 결과적으로 블록체인 기반의 투표 시스템은 공정하고 신뢰할 수 있는 선거 환경을 조성할 잠재력을 지닌다. 각 투표는 변경 불가능한 기록으로 저장되어 조작의 가능성을 원천적으로 차단한다.


In [7]:
def calc_ppl(text):
    """
    텍스트의 Perplexity 계산
    낮을수록 = 모델이 자연스럽다고 판단
    """
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])

    ppl = torch.exp(outputs.loss).item()
    return ppl


def find_best_order(sentences, separator):
    """
    24가지 순열 중 PPL 최소 순서 반환
    """
    best_ppl   = float('inf')
    best_order = None

    for perm in permutations(range(4)):
        text = separator.join([sentences[i] for i in perm])
        ppl  = calc_ppl(text)

        if ppl < best_ppl:
            best_ppl   = ppl
            best_order = list(perm)

    return best_order

In [8]:
# 전체 돌리기 전에 5개만 먼저 테스트
print("=== 소규모 테스트 (5개) ===\n")

correct = 0
for i in range(5):
    row       = train_df.iloc[i]
    sentences = [row['sentence_0'], row['sentence_1'],
                 row['sentence_2'], row['sentence_3']]
    answer    = [int(row['answer_0']), int(row['answer_1']),
                 int(row['answer_2']), int(row['answer_3'])]

    pred = find_best_order(sentences, SEPARATOR)

    is_correct = (pred == answer)
    if is_correct:
        correct += 1

    print(f"샘플 {i+1}")
    print(f"  예측: {pred}")
    print(f"  정답: {answer}")
    print(f"  결과: {'✅' if is_correct else '❌'}")

print(f"\n5개 중 {correct}개 정답")

=== 소규모 테스트 (5개) ===

샘플 1
  예측: [0, 3, 1, 2]
  정답: [0, 3, 1, 2]
  결과: ✅
샘플 2
  예측: [0, 2, 3, 1]
  정답: [0, 3, 2, 1]
  결과: ❌
샘플 3
  예측: [3, 1, 2, 0]
  정답: [3, 2, 1, 0]
  결과: ❌
샘플 4
  예측: [2, 0, 1, 3]
  정답: [2, 0, 1, 3]
  결과: ✅
샘플 5
  예측: [1, 3, 0, 2]
  정답: [1, 3, 0, 2]
  결과: ✅

5개 중 3개 정답


In [12]:
import os
import ast

print("=== train 데이터로 정확도 검증 ===\n")

# 실험마다 고유한 중간 저장 파일명
EXP_NAME = f"{MODEL_NAME.split('/')[-1]}_{repr(SEPARATOR).replace(' ', 'space')}"
temp_path = f'/content/drive/MyDrive/sentence_prediction/results/temp_{EXP_NAME}.csv'

# 이미 저장된 중간 결과 있으면 불러오기
if os.path.exists(temp_path):
    temp_df = pd.read_csv(temp_path)
    train_predictions = temp_df['pred'].tolist()
    start_idx = len(train_predictions)
    print(f"이어서 시작: {start_idx}번째부터")
else:
    train_predictions = []
    start_idx = 0
    print("처음부터 시작")

# start_idx부터 이어서 실행
for i, (_, row) in enumerate(tqdm(
    train_df.iloc[start_idx:].iterrows(),
    total=len(train_df) - start_idx
)):
    sentences = [row['sentence_0'], row['sentence_1'],
                 row['sentence_2'], row['sentence_3']]
    pred = find_best_order(sentences, SEPARATOR)
    train_predictions.append(str(pred))

    # 100개마다 중간 저장
    if (start_idx + i + 1) % 100 == 0:
        pd.DataFrame({'pred': train_predictions}).to_csv(temp_path, index=False)

# 정확도 계산
correct = 0
for i, pred_str in enumerate(train_predictions):
    pred   = ast.literal_eval(pred_str)
    answer = [int(train_df.iloc[i]['answer_0']),
              int(train_df.iloc[i]['answer_1']),
              int(train_df.iloc[i]['answer_2']),
              int(train_df.iloc[i]['answer_3'])]
    if pred == answer:
        correct += 1

acc = correct / len(train_df)
print(f"\n모델     : {MODEL_NAME}")
print(f"연결방식 : {repr(SEPARATOR)}")
print(f"정확도   : {acc:.4f} ({correct}/{len(train_df)})")

# 실험 결과 누적 저장
result_df   = pd.DataFrame({
    'model'    : [MODEL_NAME],
    'separator': [repr(SEPARATOR)],
    'accuracy' : [acc]
})

result_path = '/content/drive/MyDrive/sentence_prediction/results/experiments.csv'

if os.path.exists(result_path):
    existing  = pd.read_csv(result_path)
    result_df = pd.concat([existing, result_df], ignore_index=True)

result_df.to_csv(result_path, index=False)

# 완료 후 중간 저장 파일 삭제 (정리)
if os.path.exists(temp_path):
    os.remove(temp_path)
    print("중간 저장 파일 삭제 완료")

print("결과 저장 완료!")
print(result_df)

=== train 데이터로 정확도 검증 ===

처음부터 시작


100%|██████████| 7351/7351 [2:46:59<00:00,  1.36s/it]



모델     : kakaocorp/kanana-nano-2.1b-base
연결방식 : ' '
정확도   : 0.7628 (5607/7351)
중간 저장 파일 삭제 완료
결과 저장 완료!
                             model separator  accuracy
0               skt/kogpt2-base-v2       ' '  0.031560
1  kakaocorp/kanana-nano-2.1b-base       ' '  0.762753
